# Without XAI — Varying train/pred sample sizes

Runs all models for each sample size in `N_SAMPLES_LIST`. Results saved to separate JSON files.

In [ ]:
import pandas as pd
import numpy as np
import json
import time
from openai import OpenAI
import httpx
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
REQUEST_TIMEOUT = 900.0

models = [
    {"name": "glm-4.7-flash:latest",  "size": "medium"},
    {"name": "qwen3:14b",              "size": "medium"},
    {"name": "gpt-oss:20b",            "size": "medium"},
    {"name": "qwen3:30b",              "size": "large"},
]

N_SAMPLES_LIST = [10, 20, 40]

# Leitura e Pre-processamento dos Dados

In [ ]:
df = pd.read_csv("../Network_logs.csv")

In [ ]:
networkData = df.copy()
networkData.drop(['Source_IP', 'Destination_IP', 'Intrusion'], axis=1, inplace=True)
networkData.head(5)

In [ ]:
categorical_cols = ['Request_Type', 'Protocol', 'User_Agent', 'Status', 'Port']
for col in categorical_cols:
    networkData[col] = networkData[col].astype('category')

for col in categorical_cols:
    print(f"{col} categories: {networkData[col].cat.categories.tolist()}")

for col in categorical_cols:
    networkData[col] = networkData[col].cat.codes

In [ ]:
target_encoder = LabelEncoder()
networkData['Scan_Type_Label'] = target_encoder.fit_transform(networkData['Scan_Type'])

label_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print("Label Mapping:", label_mapping)

networkData.drop(['Scan_Type'], axis=1, inplace=True)
networkData.head(5)

In [ ]:
scaler = StandardScaler()
networkData['Payload_Size'] = scaler.fit_transform(networkData[['Payload_Size']])

X = networkData.drop(['Scan_Type_Label'], axis=1)
y = networkData['Scan_Type_Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)
y_train = pd.Series(y_train.values.ravel(), name='Scan_Type_Label')

print('SMOTE aplicado com sucesso.\n')
print('Nova distribuição:\n')
print(y_train.value_counts())

# Treinamento do Modelo

In [ ]:
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Acurácia: {acc:.4f}")
print(classification_report(y_test, y_pred))

# Helpers

In [ ]:
feature_names = list(X.columns)
class_names = list(target_encoder.classes_)

column_description = {
    "Port": "Communication port (encoded: 21=0, 22=1, 23=2, 25=3, 53=4, 80=5, 135=6, 443=7, 4444=8, 6667=9, 8080=10, 31337=11)",
    "Request_Type": "Request type (DNS=0, FTP=1, HTTP=2, HTTPS=3, SMTP=4, SSH=5, Telnet=6)",
    "Protocol": "Transport protocol (ICMP=0, TCP=1, UDP=2)",
    "Payload_Size": "Packet payload size (StandardScaler normalized)",
    "User_Agent": "Client agent (Mozilla/5.0=0, Nikto/2.1.6=1, Wget/1.20.3=2, curl/7.68.0=3, nmap/7.80=4, python-requests/2.25.1=5)",
    "Status": "Request status (Failure=0, Success=1)",
    "Scan_Type_Label": "Target variable: BotAttack=0, Normal=1, PortScan=2"
}

model_info = {
    "type": "Random Forest",
    "task": "Network intrusion detection from log entries",
    "target": "Scan_Type_Label",
    "classes": {"BotAttack": 0, "Normal": 1, "PortScan": 2},
    "features": list(X.columns),
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "pipeline": "Data cleaning → dropped IP columns → label encoding of categoricals → StandardScaler on Payload_Size → SMOTE class balancing → 70/30 stratified train/test split"
}

system_msg = (
    "You are an expert in Machine Learning, Explainable AI, and Cybersecurity. "
    "You will receive data about a network intrusion detection model that has already been "
    "fully validated through a complete data science pipeline (data cleaning, feature engineering, "
    "label encoding, SMOTE class balancing, and stratified train/test split). "
    "The model achieves high accuracy and is considered reliable. "
    "Your task is NOT to evaluate the model's predictive performance — that is already established. "
    "Your task is to EXPLAIN the model's behavior: why it makes the decisions it does, "
    "which features drive predictions, and what insights a SOC analyst can extract. "
    "After each message, briefly summarize the key data you received (do NOT analyze yet). "
    "When you receive 'OK, start your analysis.', produce your full structured analysis "
    "using ALL data from ALL previous messages."
)


def build_chat_chunks(n_samples):
    """Build the multi-turn chat messages for a given sample size."""
    train_sample = X_train.sample(n_samples, random_state=42)
    train_sample["Scan_Type_Label"] = y_train.loc[train_sample.index]
    train_sample_json = train_sample.to_json(orient="records")

    pred_sample = pd.DataFrame({"real": y_test, "predicted": y_pred})
    pred_sample_json = pred_sample.sample(n_samples, random_state=42).to_json(orient="records")

    msg_model_info = f"""I will send you data about a network intrusion detection model in parts.
After each part, briefly summarize what you received. Do NOT analyze yet.

# Part 1: Model Info & Column Descriptions

## Model
{json.dumps(model_info, indent=2, ensure_ascii=False)}

## Column Descriptions (all features are numerically encoded)
{json.dumps(column_description, indent=2, ensure_ascii=False)}

**Important:** This model has been through a complete data science pipeline (cleaning, encoding,
balancing, validation). It achieves {model_info['accuracy']*100:.1f}% accuracy on the full test set.
The model is already validated — your goal is to EXPLAIN its behavior, not to evaluate its performance."""

    msg_train = f"""# Part 2: Representative Data Examples ({n_samples} records)

These are representative examples from the training data to help you understand the feature space
and data distribution. They are NOT the evaluation set.
{train_sample_json}"""

    msg_pred = f"""# Part 3: Sample of Real vs Predicted Labels ({n_samples} records)

These are a small sample from the test set to illustrate prediction patterns.
The model achieves {model_info['accuracy']*100:.1f}% accuracy on the full test set ({len(y_test)} records).
{pred_sample_json}"""

    analysis_task = f"""# OK, start your analysis.

You have received:
- Model info (Random Forest, 3-class intrusion detection, {model_info['accuracy']*100:.1f}% accuracy) + column descriptions
- {n_samples} representative training examples (to understand the feature space)
- {n_samples} sample predictions (the model achieves {model_info['accuracy']*100:.1f}% on the full test set)

The model is already validated and performs well. Focus your analysis on EXPLAINING the model's
behavior, not on evaluating its accuracy. Provide a detailed, structured analysis:

1. **Model Context:** Briefly acknowledge the model's performance (accuracy, per-class balance)
   as context for the explanation that follows. Do not spend effort analyzing accuracy — it is
   already validated. Just establish the baseline so the rest of your analysis has credibility.

2. **Feature Analysis:** Which features are most important for distinguishing
   BotAttack vs Normal vs PortScan? Rank them and explain why from a cybersecurity perspective.

3. **Decision Pattern Hypotheses:** Based on the data patterns, hypothesize how the model
   distinguishes between attack types. What feature combinations likely drive each class prediction?

4. **Instance-Level Reasoning:** For specific instances in the data, explain what likely drove
   each prediction. What features were decisive?

5. **Cybersecurity Insights:** What are the strongest indicators of BotAttack vs PortScan?
   How could a SOC analyst use these patterns in practice?

6. **Model Behavior Assessment:** Based on the data patterns, what are the model's decision
   boundaries? Where might it struggle to distinguish between classes?

7. **Improvement Suggestions:** Concrete improvements you would recommend to make the model
   more interpretable or robust.

Use numbered sections and subsections."""

    return [
        {"user": msg_model_info},
        {"user": msg_train},
        {"user": msg_pred},
        {"user": analysis_task, "is_analysis": True},
    ]


def run_chat(client, model_name, model_size, chat_chunks):
    """Run a multi-turn chat and return the final analysis text."""
    messages = [{"role": "system", "content": system_msg}]
    max_tokens_analysis = 16384 if model_size == "large" else 8192
    n_msgs = len(chat_chunks)

    for i, chunk in enumerate(chat_chunks):
        is_analysis = chunk.get("is_analysis", False)
        messages.append({"role": "user", "content": chunk["user"]})

        if not is_analysis:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                max_tokens=256
            )
            summary = response.choices[0].message.content.strip()
            messages.append({"role": "assistant", "content": summary})
            print(f"    Msg {i+1}/{n_msgs} -> {summary[:80]}")
        else:
            print(f"    Msg {i+1}/{n_msgs} -> requesting analysis...")
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                max_tokens=max_tokens_analysis
            )
            result = response.choices[0].message.content
            print(f"    Analysis done: {len(result)} chars")
            return result

    return "ERROR: no analysis message found"


client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
    timeout=httpx.Timeout(REQUEST_TIMEOUT, connect=30.0),
)

# Execução — Loop over sample sizes

In [ ]:
all_results = {}  # {n_samples: {model_name: analysis_text}}
all_timings = {}  # {n_samples: {model_name: elapsed}}
previous_model = None

for n_samples in N_SAMPLES_LIST:
    print(f"\n{'#'*60}")
    print(f"# N_SAMPLES = {n_samples}")
    print(f"{'#'*60}")

    chat_chunks = build_chat_chunks(n_samples)
    for i, chunk in enumerate(chat_chunks, 1):
        print(f"  Msg {i}: {len(chunk['user'])} chars {'[ANALYSIS]' if chunk.get('is_analysis') else ''}")

    results = {}
    timings = {}

    for model_cfg in models:
        model_name = model_cfg["name"]
        model_size = model_cfg["size"]

        if previous_model:
            !ollama stop $previous_model
            time.sleep(2)

        print(f"\n{'='*60}")
        print(f"Modelo: {model_name} (tier: {model_size}) | N_SAMPLES={n_samples}")
        print(f"{'='*60}")

        t_start = time.time()
        try:
            results[model_name] = run_chat(client, model_name, model_size, chat_chunks)
        except Exception as e:
            results[model_name] = f"ERROR: {str(e)}"
            print(f"  Error: {e}")

        elapsed = time.time() - t_start
        timings[model_name] = elapsed
        print(f"  Time: {elapsed:.1f}s")
        previous_model = model_name

    all_results[n_samples] = results
    all_timings[n_samples] = timings

    # Save results for this sample size
    out_file = f"resultados_without_xai_samples_{n_samples}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"\nResultados salvos em {out_file}")

if previous_model:
    !ollama stop $previous_model

print("\n" + "="*60)
print("All runs completed.")
print("="*60)

# Resultados

In [ ]:
for n_samples, results in all_results.items():
    print(f"\n{'#'*60}")
    print(f"# N_SAMPLES = {n_samples}")
    print(f"{'#'*60}")
    for model_name, result in results.items():
        print(f"\n{'='*60}")
        print(f"MODELO: {model_name}")
        print(f"{'='*60}\n")
        print(result)
        print()

print("\nTimings:")
for n_samples, timings in all_timings.items():
    print(f"\n  N_SAMPLES={n_samples}:")
    for name, t in timings.items():
        print(f"    {name}: {t:.1f}s")

In [ ]:
for n_samples in N_SAMPLES_LIST:
    out_file = f"resultados_without_xai_samples_{n_samples}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(all_results[n_samples], f, ensure_ascii=False, indent=2)
    print(f"Resultados salvos em {out_file}")